In [2]:
# %pip install -q openai
# %pip install -q google-genai jsonschema
# %pip install -q pydantic

In [3]:
from dotenv import load_dotenv
import os, getpass
from openai import OpenAI
# os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
# os.environ["GEMINI_API_KEY"] = getpass.getpass("GEMINI_API_KEY: ")

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

## define PRODUCTS

In [4]:
PRODUCTS = {
  "P1": {"product_name": "Weekend city trip",
         "facts_line": "Two nights in a central hotel with breakfast, flexible dates, and free cancellation up to 48 hours before check-in."},
  "P2": {"product_name": "Fitness app",
         "facts_line": "Personalized workouts from 10–30 minutes, progress tracking, and adjustable plans for strength, cardio, or mobility."},
  "P3": {"product_name": "Noise-cancelling headphones",
         "facts_line": "Active noise cancellation, transparency mode, 30-hour battery, and comfortable over-ear fit for travel or focus."},
  "P4": {"product_name": "Language-learning course",
         "facts_line": "Structured lessons, speaking practice, feedback exercises, and weekly goals designed for steady progress."},
  "P5": {"product_name": "Subscription coffee",
         "facts_line": "Fresh beans delivered monthly, your roast preference, flexible skip/pause, and tasting notes included with each delivery."},
  "P6": {"product_name": "Productivity tool",
         "facts_line": "Task lists, calendars, reminders, and templates that sync across devices, with simple sharing for projects."},
}

## Pydantic schema (our contract)

In [5]:


from pydantic import BaseModel, Field
from typing import List, Literal

class Meta(BaseModel):
    product_id: str
    product_name: str
    trait_axis: Literal["EXTRAVERSION", "OPENNESS"]
    variant: Literal["E_PLUS", "E_MINUS", "O_PLUS", "O_MINUS"]
    paraphrase_id: int = Field(ge=1, le=3)
    word_target: int
    word_range: List[int]

class Constraints(BaseModel):
    facts_line_must_match_verbatim: bool
    no_personality_labels: bool
    no_extra_benefits_or_claims: bool
    no_social_proof_scarcity_authority: bool
    cta_strength_equal_across_variants: bool

class Content(BaseModel):
    hook: str
    facts: str
    framing: str
    cta: str
    full_message: str

class SelfChecks(BaseModel):
    approx_word_count_full_message: int
    facts_line_verbatim_confirmed: bool
    forbidden_items_present: List[str]

class Stimulus(BaseModel):
    meta: Meta
    constraints: Constraints
    content: Content
    self_checks: SelfChecks

## Gemini call (LLM instructions)

In [6]:
import os, json, re
from google import genai
from google.genai import types

client = genai.Client()

WORD_MIN, WORD_MAX = 90, 110

def wc(text: str) -> int:
    return len(re.findall(r"\b[\w’'-]+\b", text))

def build_input(product_id: str, variant: str, paraphrase_id: int, PRODUCTS: dict) -> str:
    p = PRODUCTS[product_id]
    trait_axis = "EXTRAVERSION" if variant.startswith("E_") else "OPENNESS"
    return f"""Generate ONE ad message stimulus.

META:
- product_id: {product_id}
- product_name: {p["product_name"]}
- trait_axis: {trait_axis}
- variant: {variant}
- paraphrase_id: {paraphrase_id}
- word_target: 100
- word_range: [{WORD_MIN}, {WORD_MAX}]

FACTS line (must match verbatim exactly):
{p["facts_line"]}
"""

GENERATOR_INSTRUCTIONS = f"""
You are a stimulus generator for a behavioral experiment.

Hard constraints:
- full_message length: {WORD_MIN}–{WORD_MAX} words.
- The FACTS sentence must be reproduced verbatim EXACTLY as provided.
- Do NOT add benefits/claims beyond the FACTS sentence (no discounts, guarantees, testimonials).
- Do NOT use social proof, scarcity/urgency, or authority cues.
- Do NOT mention personality labels.
- Targeting differences must appear ONLY in hook/framing; facts must stay identical.

Structure inside content:
- hook: 1 sentence
- facts: 1 sentence (verbatim)
- framing: 2–3 sentences (targeting happens here)
- cta: 1 neutral sentence (no urgency)
- full_message: hook + facts + framing + cta concatenated with single spaces.

Variant targeting rules:
- E_PLUS: social orientation + energetic/action tone (>=2 social cues, >=2 high-energy cues).
- E_MINUS: calm autonomy/solo tone (>=2 autonomy cues, >=2 low-arousal cues; avoid explicit social words).
- O_PLUS: novelty/exploration + abstract/value framing (>=2 novelty cues, >=2 abstract/value cues).
- O_MINUS: conventional/practical/predictable + concrete utility (>=2 predictability cues, >=2 utility cues; avoid novelty language).
""".strip()

def generate_one(product_id, variant, paraphrase_id, PRODUCTS, model="gemini-2.5-flash"):
    prompt = GENERATOR_INSTRUCTIONS + "\n\n" + build_input(product_id, variant, paraphrase_id, PRODUCTS)

    resp = client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.8,
            response_mime_type="application/json",
            response_schema=Stimulus,   # <-- schema-enforced JSON
        ),
    )

    # Guard: empty output
    if not getattr(resp, "text", None) or not resp.text.strip():
        # Print minimal diagnostics (finish reason / safety may explain empties)
        cand = resp.candidates[0] if getattr(resp, "candidates", None) else None
        raise RuntimeError(
            f"Empty response.text. "
            f"finish_reason={getattr(cand, 'finish_reason', None)} "
            f"safety_ratings={getattr(cand, 'safety_ratings', None)}"
        )

    # Parse + validate via Pydantic
    try:
        stim = Stimulus.model_validate_json(resp.text)
    except Exception as e:
        import json as _json
        _raw = getattr(resp, "text", None)
        try:
            _parsed = _json.loads(_raw) if _raw else None
        except Exception as je:
            raise RuntimeError(f"Validation failed and response not valid JSON. resp.text={_raw!r}; json error: {je}") from e
        raise RuntimeError(f"Validation failed. Parsed JSON top-level keys: {list(_parsed.keys()) if isinstance(_parsed, dict) else type(_parsed)}; resp.text={_raw!r}") from e

    # Extra hard checks you care about (facts + word count)
    expected = PRODUCTS[product_id]["facts_line"].strip()
    if stim.content.facts.strip() != expected:
        raise ValueError("FACTS line mismatch (must be verbatim).")

    # Prefer the model's self-check word count when available
    count = getattr(stim.self_checks, "approx_word_count_full_message", None)
    if count is None:
        count = wc(stim.content.full_message)

    if not (WORD_MIN <= count <= WORD_MAX):
        _raw = getattr(resp, "text", None)
        raise ValueError(f"Word count out of range: {count}. resp.text={_raw!r}")

    return stim

## Generate 1 stimulus

In [7]:
try:
    stim = generate_one("P1", "E_PLUS", 1, PRODUCTS)
    print(stim.content.full_message)
except Exception:
    import traceback, sys
    traceback.print_exc()
    # also show repr of exception to preserve newlines
    import builtins
    print("\n--- repr(exception) ---")
    print(repr(sys.exc_info()[1]))

Ready to ignite your next adventure with friends? Two nights in a central hotel with breakfast, flexible dates, and free cancellation up to 48 hours before check-in. Gather your favorite people and dive into the vibrant pulse of the city. Explore bustling streets, share laughter over delicious meals, and create unforgettable memories together. This trip is designed for dynamic group experiences, ensuring everyone can participate in the excitement. Plan your exciting city getaway today.


In [8]:
from pathlib import Path
out_dir = Path('stimuli')
out_dir.mkdir(parents=True, exist_ok=True)
try:
    message = stim.content.full_message
except NameError:
    stim = generate_one('P1', 'E_PLUS', 1, PRODUCTS)
    message = stim.content.full_message
out_path = out_dir / 'gemini-messages.txt'
with open(out_path, 'a', encoding='utf-8') as f:
    f.write(message.replace('\n', ' ') + '\n\n')
print(f'Appended message to {out_path}')

Appended message to stimuli\gemini-messages.txt
